In [8]:
import cv2
import os
import shutil
import json
import insightface as inf

input_folders = ["JurassicPark720p"]
master_folder = 'faces_detected'

# Create the master folder if it doesn't exist
os.makedirs(master_folder, exist_ok=True)
# Path to save the JSON file
output_json_file = 'faces_data.json'


In [5]:
# Load the Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Dictionary to store the face detection data
if os.path.exists(output_json_file):
    with open(output_json_file, 'r') as f:
        faces_data = json.load(f)
else:
    faces_data = {}

def count_faces_in_video(video_path):
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    total_faces = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert the frame to grayscale for face detection
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # Detect faces in the frame
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
        
        # Add the number of faces detected in this frame to the total count
        total_faces += len(faces)
    
    cap.release()
    return total_faces

In [9]:
# Iterate over each folder in the input_folders list
for folder in input_folders:
    movie_name = os.path.basename(folder)  # Use the folder name as the movie name
    faces_data[movie_name] = []  # Initialize an empty list to store clip data for the current movie
    
    # Iterate over files in the current folder
    for filename in os.listdir(folder):
        if filename.endswith('.mp4'): 
            video_path = os.path.join(folder, filename)
            
            face_count = count_faces_in_video(video_path)
            
            if face_count > 0:
                # Copy the video to the master folder
                shutil.copy2(video_path, os.path.join(master_folder, filename))
                
                # Store the data in the faces_data dictionary
                faces_data[movie_name].append({
                    "clip_name": filename,
                    "num_faces": face_count
                })
    # Write the faces data to the output JSON file after every movie
    with open(output_json_file, 'w') as json_file:
        json.dump(faces_data, json_file, indent=4)



print(f'Processing complete. Face data saved to {output_json_file}.')


KeyboardInterrupt: 

In [9]:

# Initialize the InsightFace app
app = inf.App()

def count_faces_in_video_insightface(video_path):
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    total_faces = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Detect faces using InsightFace
        faces = app.get(frame)
        
        # Add the number of faces detected in this frame to the total count
        total_faces += len(faces)
    
    cap.release()
    return total_faces

# Update the face counting process to use InsightFace
for folder in input_folders:
    movie_name = os.path.basename(folder)  # Use the folder name as the movie name
    faces_data[movie_name] = []  # Initialize an empty list to store clip data for the current movie
    
    # Iterate over files in the current folder
    for filename in os.listdir(folder):
        if filename.endswith('.mp4'): 
            video_path = os.path.join(folder, filename)
            
            face_count = count_faces_in_video_insightface(video_path)
            
            if face_count > 0:
                # Copy the video to the master folder
                shutil.copy2(video_path, os.path.join(master_folder, filename))
                
                # Store the data in the faces_data dictionary
                faces_data[movie_name].append({
                    "clip_name": filename,
                    "num_faces": face_count
                })
    # Write the faces data to the output JSON file after every movie
    with open(output_json_file, 'w') as json_file:
        json.dump(faces_data, json_file, indent=4)

print(f'Processing complete with InsightFace. Face data saved to {output_json_file}.')



AttributeError: module 'insightface' has no attribute 'App'